In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "full"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage7"
SEED = 913
MODEL_NAME = ["jepa_wm_pusht", "jepa_wm_wall"]
ENVIRONMENT = ["PushT", "Wall"]
HORIZONS = [1, 3, 6]
NUM_STATES = 24  # per environment in smoke mode
ACTIONS_PER_STATE = 10

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage7"
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
TARGET_STEPS = list(range(1, max(HORIZONS) + 1))
TASKS_PER_ENVIRONMENT = 12
TASK_SPLIT_COUNTS = [6, 3, 0, 3]
EVALUATION_SEEDS = [913, 1297, 1709]
RIDGE_LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
BOOTSTRAP_REPS = 200
RANKING_TIE = 1e-9

AUDIT_PROJECTION_DIM = 64
AUDIT_PROJECTION_SEEDS = [7101]
ADAPTER_SEEDS = [7201]
ADAPTER_METHODS = [
    "absolute_residual",
    "independent_delta_control",
    "counterfactual_recurrent",
]
ADAPTER_HIDDEN_DIM = 64
ADAPTER_IMPLEMENTATION_ID = "recurrent_token_delta_v1"
TRAINING_EPOCHS = 4
SELECTION_EPOCHS = [2, 4]
TRAINING_BATCH_STATES = 2
TRAINING_LR = 3e-4
TRAINING_WEIGHT_DECAY = 1e-4
COUNTERFACTUAL_WEIGHT = 1.0
LATENT_ERROR_RATIO_MARGIN = 1.10

DOWNLOAD_RESULTS = True
EVIDENCE_STATUS = "EXPLORATORY_DEVELOPMENT"
TASK_FAMILY_ID = "stage5_tasks_reused_for_stage7_development"
DEVELOPMENT_SPLIT = "development_holdout"

if RUN_MODE == "full":
    NUM_STATES = 96
    AUDIT_PROJECTION_DIM = 128
    AUDIT_PROJECTION_SEEDS = [7101, 9101]
    ADAPTER_SEEDS = [7201, 9201]
    TRAINING_EPOCHS = 30
    SELECTION_EPOCHS = [10, 20, 30]
    TRAINING_BATCH_STATES = 4
    BOOTSTRAP_REPS = 2000
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == ["jepa_wm_pusht", "jepa_wm_wall"]
assert ENVIRONMENT == ["PushT", "Wall"]
assert HORIZONS == [1, 3, 6]
assert TARGET_STEPS == [1, 2, 3, 4, 5, 6]
assert ACTIONS_PER_STATE == 10
assert NUM_STATES % TASKS_PER_ENVIRONMENT == 0
assert sum(TASK_SPLIT_COUNTS) == TASKS_PER_ENVIRONMENT
assert SELECTION_EPOCHS[-1] == TRAINING_EPOCHS


# Stage 7: recurrent counterfactual transition adapter

Stage 6 trained terminal readouts after the complete JEPA-WM rollout and found
that short-horizon action-effect supervision did not survive reliably to
horizon six. Stage 7 tests the inference-path hypothesis directly.

The notebook first retains the unpooled 16×16 visual tokens and audits physical
effect decodability after every one of the six AdaLN predictor blocks and every
rollout step. It then trains a small shared token residual using true
same-state, no-op-relative latent transition differences. The selected
residual is inserted after every `forward_pred` call *before* that prediction
is recycled as the next context.

No ranking gradient enters the dynamics. The public DINO encoder and JEPA-WM
predictor remain frozen; only the recurrent residual is trained. Endpoint
ranking uses the model's native latent goal distance rather than a pose
decoder.

This is exploratory development. It reuses previously inspected Stage 5 tasks,
renames the former final split `development_holdout`, and cannot support a
confirmatory efficacy claim. A positive result nominates one frozen recipe for
new numerical tasks.


In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-matched torch and torchvision builds.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected.")


In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as torch_functional
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
INTERMEDIATE = OUT / "intermediate"
TRUTH_ROOT = INTERMEDIATE / "truth"
MODEL_ROOT = INTERMEDIATE / "models"
LOG_DIR = OUT / "logs"
PLOT_DIR = OUT / "plots"
PROBE_DIR = OUT / "probes"
for path in [
    OUT,
    INTERMEDIATE,
    TRUTH_ROOT,
    MODEL_ROOT,
    LOG_DIR,
    PLOT_DIR,
    PROBE_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage7")
log.info("Seeds set to %d; evaluation seeds=%s; adapter seeds=%s", SEED, EVALUATION_SEEDS, ADAPTER_SEEDS)


def gpu_report(label):
    payload = {
        "label": label,
        "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    }
    log.info("GPU memory %s", payload)
    return payload


CONFIG = {
    "RUN_MODE": RUN_MODE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SEED": SEED,
    "MODEL_NAME": MODEL_NAME,
    "ENVIRONMENT": ENVIRONMENT,
    "HORIZONS": HORIZONS,
    "NUM_STATES": NUM_STATES,
    "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "REPO_URL": REPO_URL,
    "REPO_COMMIT": REPO_COMMIT,
    "FRAMESKIP": FRAMESKIP,
    "TARGET_STEPS": TARGET_STEPS,
    "TASKS_PER_ENVIRONMENT": TASKS_PER_ENVIRONMENT,
    "TASK_SPLIT_COUNTS": TASK_SPLIT_COUNTS,
    "EVALUATION_SEEDS": EVALUATION_SEEDS,
    "RIDGE_LAMBDAS": RIDGE_LAMBDAS,
    "BOOTSTRAP_REPS": BOOTSTRAP_REPS,
    "RANKING_TIE": RANKING_TIE,
    "AUDIT_PROJECTION_DIM": AUDIT_PROJECTION_DIM,
    "AUDIT_PROJECTION_SEEDS": AUDIT_PROJECTION_SEEDS,
    "ADAPTER_SEEDS": ADAPTER_SEEDS,
    "ADAPTER_METHODS": ADAPTER_METHODS,
    "ADAPTER_HIDDEN_DIM": ADAPTER_HIDDEN_DIM,
    "ADAPTER_IMPLEMENTATION_ID": ADAPTER_IMPLEMENTATION_ID,
    "TRAINING_EPOCHS": TRAINING_EPOCHS,
    "SELECTION_EPOCHS": SELECTION_EPOCHS,
    "TRAINING_BATCH_STATES": TRAINING_BATCH_STATES,
    "TRAINING_LR": TRAINING_LR,
    "TRAINING_WEIGHT_DECAY": TRAINING_WEIGHT_DECAY,
    "COUNTERFACTUAL_WEIGHT": COUNTERFACTUAL_WEIGHT,
    "LATENT_ERROR_RATIO_MARGIN": LATENT_ERROR_RATIO_MARGIN,
    "DOWNLOAD_RESULTS": DOWNLOAD_RESULTS,
    "EVIDENCE_STATUS": EVIDENCE_STATUS,
    "TASK_FAMILY_ID": TASK_FAMILY_ID,
    "DEVELOPMENT_SPLIT": DEVELOPMENT_SPLIT,
    "pinned_dependencies": PINNED,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()
CONFIG["run_signature"] = RUN_SIGNATURE
config_path = OUT / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    if previous.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "OUTPUT_DIR contains a different configuration; choose a new OUTPUT_DIR."
        )
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
(OUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

PIPELINE_FAILED = False
FAILURE_MESSAGE = ""


def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)


gpu_report("startup")


In [ ]:
MODEL_BY_ENVIRONMENT = {
    "PushT": ["jepa_wm_pusht"],
    "Wall": ["jepa_wm_wall"],
}
READOUTS = [
    "latent_distance",
    "linear_pose",
    "action_blind",
    "linear_pose_shuffled",
    "oracle_pose",
]
SPLIT_NAMES = [
    "probe_train",
    "probe_calibration",
    "regression_train",
    DEVELOPMENT_SPLIT,
]


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=True) + "\n")


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        if not rows:
            raise ValueError(f"cannot infer fields for empty table {path}")
        fieldnames = list(rows[0])
    temporary = Path(path).with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def pair_indices(n_actions):
    return np.triu_indices(n_actions, k=1)


def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim != 5:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}


def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError("zero direction")
    return vector / norm


def rotate_vector(vector, degrees):
    radians = np.deg2rad(degrees)
    return np.array(
        [
            [np.cos(radians), -np.sin(radians)],
            [np.sin(radians), np.cos(radians)],
        ]
    ) @ np.asarray(vector)


def feature_metrics(truth, prediction, eps=1e-12):
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(0, 2)))
    common = np.mean(errors, axis=0)
    centered = errors - common[None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(0, 2)))
    left, right = pair_indices(truth.shape[0])
    truth_delta = truth[left] - truth[right]
    predicted_delta = prediction[left] - prediction[right]
    pair_error = predicted_delta - truth_delta
    pair_rmse = np.sqrt(np.mean(pair_error**2, axis=(0, 2)))
    pair_scale = np.sqrt(np.mean(truth_delta**2, axis=(0, 2)))
    normalized = pair_rmse / np.maximum(pair_scale, eps)
    dot = np.sum(truth_delta * predicted_delta, axis=-1)
    denominator = (
        np.linalg.norm(truth_delta, axis=-1)
        * np.linalg.norm(predicted_delta, axis=-1)
    )
    cosine = np.divide(
        dot,
        denominator,
        out=np.zeros_like(dot),
        where=denominator > eps,
    ).mean(axis=0)
    expected_pair_mse = (
        2 * truth.shape[0] / (truth.shape[0] - 1)
    ) * action_dependent**2
    return {
        "ordinary_feature_rmse": ordinary,
        "common_mode_feature_rmse": np.sqrt(np.mean(common**2, axis=-1)),
        "action_dependent_feature_rmse": action_dependent,
        "paired_feature_rmse": pair_rmse,
        "normalized_paired_feature_rmse": normalized,
        "paired_feature_cosine": cosine,
        "pair_identity_residual": pair_rmse**2 - expected_pair_mse,
    }


def ranking_metrics(true_cost, predicted_cost, tie=RANKING_TIE):
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    regret = chosen - best
    normalized_regret = regret / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    pairwise = float(np.nanmean(credit)) if np.any(valid) else float("nan")
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else float("nan")
    )
    margin_scale = (
        float(np.sqrt(np.mean(true_margin[valid] ** 2))) if np.any(valid) else 0.0
    )
    normalized_margin_rmse = (
        float(
            np.sqrt(
                np.mean((predicted_margin[valid] - true_margin[valid]) ** 2)
            )
            / margin_scale
        )
        if margin_scale > tie
        else float("nan")
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "pairwise_accuracy": pairwise,
        "weighted_pairwise_accuracy": weighted,
        "normalized_margin_rmse": normalized_margin_rmse,
        "pair_left": left,
        "pair_right": right,
        "true_margin": true_margin,
        "predicted_margin": predicted_margin,
        "pair_credit": credit,
        "pair_weight": weights,
    }


def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    finite = np.isfinite(values)
    values = values[finite]
    groups = groups[finite]
    unique = np.unique(groups)
    if len(unique) == 0:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": repetitions,
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.mean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.mean(values)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }


def random_projection(input_dim, output_dim, seed):
    rng = np.random.default_rng(seed)
    projection = rng.standard_normal(
        (input_dim, output_dim)
    ).astype(np.float32)
    scale = np.float32(np.sqrt(output_dim))
    return (projection / scale).astype(np.float32, copy=False)


def standardize_fit(values):
    values = np.asarray(values, dtype=np.float64)
    mean = np.mean(values, axis=0)
    scale = np.std(values, axis=0)
    scale[scale < 1e-8] = 1.0
    return mean, scale


def fit_linear_readout(x_train, y_train, x_calibration, y_calibration):
    mean, scale = standardize_fit(x_train)
    train = (np.asarray(x_train, dtype=np.float64) - mean) / scale
    calibration = (np.asarray(x_calibration, dtype=np.float64) - mean) / scale
    train = np.column_stack([np.ones(len(train)), train])
    calibration = np.column_stack([np.ones(len(calibration)), calibration])
    gram = train.T @ train
    cross = train.T @ np.asarray(y_train, dtype=np.float64)
    best = None
    for ridge in RIDGE_LAMBDAS:
        penalty = np.eye(gram.shape[0]) * ridge
        penalty[0, 0] = 0.0
        coefficient = np.linalg.solve(gram + penalty, cross)
        prediction = calibration @ coefficient
        loss = float(np.mean((prediction - y_calibration) ** 2))
        candidate = {
            "ridge": float(ridge),
            "calibration_pose_mse": loss,
            "mean": mean,
            "scale": scale,
            "coefficient": coefficient,
        }
        if best is None or loss < best["calibration_pose_mse"]:
            best = candidate
    return best


def predict_linear_readout(probe, values):
    standardized = (
        np.asarray(values, dtype=np.float64) - probe["mean"]
    ) / probe["scale"]
    augmented = np.column_stack([np.ones(len(standardized)), standardized])
    return augmented @ probe["coefficient"]


def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT],
        check=True,
    )
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The public hub loader imports a planning entry point with extra evaluation
    # dependencies. Use its equivalent lightweight model constructor.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # PushT and Wall do not use the DROID pose helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in simulator predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    config_paths = [
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/dino-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/jepa-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in config_paths:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo


def task_split_map():
    rng = np.random.default_rng(SEED + 17)
    order = rng.permutation(TASKS_PER_ENVIRONMENT).tolist()
    mapping = {}
    start = 0
    for name, count in zip(SPLIT_NAMES, TASK_SPLIT_COUNTS):
        for task_id in order[start : start + count]:
            mapping[int(task_id)] = name
        start += count
    return mapping


def pusht_tasks():
    # Frozen before any Stage 5 simulation. None appears in the Stage 3 family.
    values = [
        (220.0, 200.0, -0.90),
        (200.0, 272.0, 0.35),
        (224.0, 320.0, -0.30),
        (270.0, 198.0, 0.95),
        (292.0, 224.0, -0.45),
        (314.0, 270.0, 0.55),
        (292.0, 316.0, -1.00),
        (246.0, 310.0, 0.15),
        (230.0, 238.0, 1.25),
        (276.0, 244.0, -1.25),
        (314.0, 292.0, 0.85),
        (244.0, 278.0, -0.75),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "PushT",
            "task_id": index,
            "task_name": f"stage5_pusht_goal_{index:02d}",
            "goal": list(value),
            "split": splits[index],
        }
        for index, value in enumerate(values)
    ]


def wall_tasks():
    # Frozen before any Stage 5 simulation. None appears in the Stage 3 family.
    layouts = [
        (26.0, 20.0, 54.0, 44.0),
        (28.0, 34.0, 54.0, 16.0),
        (30.0, 44.0, 54.0, 32.0),
        (33.0, 16.0, 10.0, 48.0),
        (35.0, 28.0, 10.0, 20.0),
        (38.0, 42.0, 10.0, 36.0),
        (25.0, 38.0, 55.0, 22.0),
        (31.0, 24.0, 55.0, 50.0),
        (36.0, 46.0, 11.0, 16.0),
        (39.0, 20.0, 11.0, 40.0),
        (29.0, 30.0, 54.0, 50.0),
        (34.0, 38.0, 10.0, 24.0),
    ]
    splits = task_split_map()
    return [
        {
            "environment": "Wall",
            "task_id": index,
            "task_name": f"stage5_wall_layout_goal_{index:02d}",
            "wall_x": wall_x,
            "door_y": door_y,
            "goal": [goal_x, goal_y],
            "split": splits[index],
        }
        for index, (wall_x, door_y, goal_x, goal_y) in enumerate(layouts)
    ]


TASKS = {"PushT": pusht_tasks(), "Wall": wall_tasks()}

# Fail before simulation if the declared split protocol and generated tasks
# ever drift apart again.
EXPECTED_TASK_SPLIT_COUNTS = {
    name: count
    for name, count in zip(SPLIT_NAMES, TASK_SPLIT_COUNTS)
    if count > 0
}
for environment, tasks in TASKS.items():
    observed = {
        name: sum(task["split"] == name for task in tasks)
        for name in {task["split"] for task in tasks}
    }
    if observed != EXPECTED_TASK_SPLIT_COUNTS:
        raise AssertionError(
            f"{environment} task split mismatch: "
            f"expected {EXPECTED_TASK_SPLIT_COUNTS}, got {observed}"
        )


def build_state_records(environment):
    tasks = TASKS[environment]
    per_task = NUM_STATES // TASKS_PER_ENVIRONMENT
    records = []
    state_id = 0
    for task in tasks:
        for within_task in range(per_task):
            evaluation_seed = EVALUATION_SEEDS[within_task % len(EVALUATION_SEEDS)]
            rng = np.random.default_rng(
                evaluation_seed * 100000
                + task["task_id"] * 1000
                + within_task
            )
            if environment == "PushT":
                goal_xy = np.asarray(task["goal"][:2], dtype=np.float64)
                for _ in range(100):
                    radial = rng.uniform(85.0, 120.0)
                    polar = rng.uniform(-np.pi, np.pi)
                    block = goal_xy + radial * np.array(
                        [np.cos(polar), np.sin(polar)]
                    )
                    direction = unit_vector(goal_xy - block)
                    agent_distance = rng.uniform(58.0, 80.0)
                    agent = block - agent_distance * direction
                    if (
                        np.all(block > 90.0)
                        and np.all(block < 422.0)
                        and np.all(agent > 35.0)
                        and np.all(agent < 477.0)
                    ):
                        break
                else:
                    raise RuntimeError("could not build bounded PushT state")
                state = np.array(
                    [
                        agent[0],
                        agent[1],
                        block[0],
                        block[1],
                        rng.uniform(-0.65, 0.65),
                        0.0,
                        0.0,
                    ],
                    dtype=np.float64,
                )
                stratum = "near" if agent_distance < 69.0 else "far"
            else:
                wall_x = float(task["wall_x"])
                goal_x = float(task["goal"][0])
                goal_on_right = goal_x > wall_x
                if goal_on_right:
                    x = rng.uniform(8.0, max(9.0, wall_x - 8.0))
                else:
                    x = rng.uniform(min(56.0, wall_x + 8.0), 57.0)
                y = rng.uniform(8.0, 57.0)
                state = np.array([x, y], dtype=np.float64)
                stratum = "left_to_right" if goal_on_right else "right_to_left"
            records.append(
                {
                    "environment": environment,
                    "state_id": state_id,
                    "task_id": task["task_id"],
                    "task_name": task["task_name"],
                    "split": task["split"],
                    "evaluation_seed": int(evaluation_seed),
                    "design_stratum": stratum,
                    "state": state,
                }
            )
            state_id += 1
    if len(records) != NUM_STATES:
        raise AssertionError("state-record count mismatch")
    return records


def pusht_candidate_library(state, task, primitive_steps):
    direction = unit_vector(
        np.asarray(task["goal"][:2]) - np.asarray(state)[2:4]
    )
    specifications = [("noop", 0.0, 0)]
    specifications.extend(
        (f"direct_{duration}", 0.0, duration)
        for duration in [8, 12, 16, 20, 24, 30]
    )
    for angle in [-20.0, 20.0]:
        for duration in [12, 18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-40.0, 40.0]:
        for duration in [18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-70.0, 70.0, 140.0, -140.0, 180.0]:
        specifications.append((f"angle_{angle:+.0f}_24", angle, 24))
    sequences = []
    for _, angle, duration in specifications:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        if duration:
            sequence[:duration] = (
                0.14 * rotate_vector(direction, angle)
            ).astype(np.float32)
        sequences.append(sequence)
    selected = np.asarray(
        [0, 2, 4, 6, 8, 11, 14, 16, 17, 18],
        dtype=np.int64,
    )
    return (
        np.stack(sequences)[selected],
        [specifications[index][0] for index in selected],
        selected,
    )


def nominal_waypoint_sequence(state, waypoints, primitive_steps, magnitude=0.75):
    position = np.asarray(state, dtype=np.float64).copy()
    sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
    waypoints = [np.asarray(point, dtype=np.float64) for point in waypoints]
    for step in range(primitive_steps):
        remaining = primitive_steps - step
        waypoint_index = min(
            len(waypoints) - 1,
            (step * len(waypoints)) // primitive_steps,
        )
        delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) < 0.5 and waypoint_index + 1 < len(waypoints):
            waypoint_index += 1
            delta = waypoints[waypoint_index] - position
        if np.linalg.norm(delta) > 1e-8:
            action = magnitude * unit_vector(delta)
            sequence[step] = action.astype(np.float32)
            position = position + 2.0 * action
    return sequence


def wall_candidate_library(state, task, primitive_steps):
    state = np.asarray(state, dtype=np.float64)
    goal = np.asarray(task["goal"], dtype=np.float64)
    wall_x = float(task["wall_x"])
    door_y = float(task["door_y"])
    side = np.sign(goal[0] - state[0])
    door = np.array([wall_x + side * 1.0, door_y])
    directions = [
        ("noop", np.zeros((primitive_steps, 2), dtype=np.float32)),
        ("direct", nominal_waypoint_sequence(state, [goal], primitive_steps)),
        (
            "via_door",
            nominal_waypoint_sequence(state, [door, goal], primitive_steps),
        ),
        (
            "via_door_high",
            nominal_waypoint_sequence(
                state,
                [door + np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
        (
            "via_door_low",
            nominal_waypoint_sequence(
                state,
                [door - np.array([0.0, 3.0]), goal],
                primitive_steps,
            ),
        ),
    ]
    direct = unit_vector(goal - state)
    for angle in [-35.0, 35.0, -70.0, 70.0]:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        sequence[:] = (0.75 * rotate_vector(direct, angle)).astype(np.float32)
        directions.append((f"angle_{angle:+.0f}", sequence))
    reverse = np.zeros((primitive_steps, 2), dtype=np.float32)
    reverse[:] = (-0.55 * direct).astype(np.float32)
    directions.append(("reverse", reverse))
    if len(directions) != ACTIONS_PER_STATE:
        raise AssertionError("Wall candidate count mismatch")
    return (
        np.stack([item[1] for item in directions]),
        [item[0] for item in directions],
        np.arange(ACTIONS_PER_STATE, dtype=np.int64),
    )


def candidate_library(environment, state, task, primitive_steps):
    if environment == "PushT":
        return pusht_candidate_library(state, task, primitive_steps)
    return wall_candidate_library(state, task, primitive_steps)


def task_cost(environment, states, task):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle_error = np.arctan2(
            np.sin(states[..., 4] - goal[2]),
            np.cos(states[..., 4] - goal[2]),
        )
        pieces = np.concatenate(
            [
                (states[..., 2:4] - goal[:2]) / 512.0,
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    goal = np.asarray(task["goal"], dtype=np.float64)
    return np.linalg.norm((states[..., :2] - goal) / 65.0, axis=-1)


def pose_target(environment, states):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        angle = states[..., 4]
        return np.stack(
            [
                states[..., 2] / 512.0,
                states[..., 3] / 512.0,
                np.sin(angle),
                np.cos(angle),
            ],
            axis=-1,
        )
    return states[..., :2] / 65.0


def decoded_task_cost(environment, prediction, task):
    prediction = np.asarray(prediction, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_error = np.arctan2(
            np.sin(angle - goal[2]),
            np.cos(angle - goal[2]),
        )
        return np.linalg.norm(
            np.concatenate(
                [
                    prediction[..., :2] - goal[:2] / 512.0,
                    (angle_error / np.pi)[..., None],
                ],
                axis=-1,
            ),
            axis=-1,
        )
    return np.linalg.norm(
        prediction[..., :2] - np.asarray(task["goal"]) / 65.0,
        axis=-1,
    )


def physical_pose_error(environment, prediction, truth):
    prediction = np.asarray(prediction, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if environment == "PushT":
        angle_prediction = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_truth = np.arctan2(truth[..., 2], truth[..., 3])
        angle_error = np.arctan2(
            np.sin(angle_prediction - angle_truth),
            np.cos(angle_prediction - angle_truth),
        )
        pieces = np.concatenate(
            [
                prediction[..., :2] - truth[..., :2],
                (angle_error / np.pi)[..., None],
            ],
            axis=-1,
        )
        return np.linalg.norm(pieces, axis=-1)
    return np.linalg.norm(prediction[..., :2] - truth[..., :2], axis=-1)


def make_environment(repo, environment, task=None):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if environment == "PushT":
        from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv

        return PushTEnv(
            with_velocity=True,
            with_target=True,
            render_size=224,
            relative=True,
            action_scale=100,
        )

    from evals.simu_env_planning.envs.wall_gym_wrap import DEFAULT_CFG
    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall

    if task is None:
        task = TASKS["Wall"][0]
    env = DotWall(
        rng=np.random.default_rng(SEED),
        wall_config=deepcopy(DEFAULT_CFG),
        fix_wall=True,
        cross_wall=False,
        device="cpu",
    )
    env.wall_x = torch.tensor(float(task["wall_x"]))
    env.hole_y = torch.tensor(float(task["door_y"]))
    env.left_wall_x = env.wall_x - env.wall_config.wall_width // 2
    env.right_wall_x = env.wall_x + env.wall_config.wall_width // 2
    return env


def wall_visual(env):
    value = env.render().float()[None]
    resized = torch_functional.interpolate(
        value,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )[0]
    return (
        torch.clamp(torch.round(resized), 0, 255)
        .to(torch.uint8)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )


def reset_environment(repo, environment, task, state, seed):
    if environment == "PushT":
        env = make_environment(repo, environment)
        env.seed(seed)
        env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
        observation, restored = env.reset()
        payload = {
            "visual": np.asarray(observation["visual"]).copy(),
            "proprio": np.asarray(observation["proprio"]).copy(),
        }
        return env, payload, np.asarray(restored).copy()

    env = make_environment(repo, environment, task)
    env.seed(seed)
    env.reset_to_state = torch.as_tensor(
        np.asarray(state, dtype=np.float32)
    )
    observation, restored = env.reset()
    payload = {
        "visual": wall_visual(env),
        "proprio": np.asarray(
            observation["proprio"].detach().cpu(), dtype=np.float32
        ),
    }
    return env, payload, np.asarray(restored.detach().cpu(), dtype=np.float32)


def rollout_branch(repo, environment, task, state, actions, seed):
    env, initial, restored = reset_environment(
        repo, environment, task, state, seed
    )
    wanted = set(TARGET_STEPS)
    observations = {}
    states = {}
    interactions = {}
    interaction_types = {}
    cumulative_interactions = 0
    cumulative_crossings = 0
    previous_state = np.asarray(restored, dtype=np.float64).copy()

    for step, action in enumerate(actions, start=1):
        if environment == "PushT":
            observation, _, _, info = env.step(action)
            current_state = np.asarray(info["state"]).copy()
            cumulative_interactions += int(info.get("n_contacts", 0))
            current_observation = {
                "visual": np.asarray(observation["visual"]).copy(),
                "proprio": np.asarray(observation["proprio"]).copy(),
            }
        else:
            observation, _, _, info = env.step(
                torch.as_tensor(action, dtype=torch.float32)
            )
            current_state = np.asarray(
                info["state"].detach().cpu(), dtype=np.float64
            )
            proposed = previous_state + 2.0 * np.asarray(action)
            if np.linalg.norm(current_state - proposed) > 1e-5:
                cumulative_interactions += 1
            wall_x = float(task["wall_x"])
            crossed = (
                (previous_state[0] - wall_x) * (current_state[0] - wall_x)
                < 0
            )
            cumulative_crossings += int(crossed)
            current_observation = {
                "visual": wall_visual(env),
                "proprio": np.asarray(
                    observation["proprio"].detach().cpu(),
                    dtype=np.float32,
                ),
            }
        previous_state = current_state.copy()
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = current_observation
                states[horizon] = current_state
                interactions[horizon] = cumulative_interactions
                if environment == "PushT":
                    interaction_types[horizon] = (
                        "contact" if cumulative_interactions > 0 else "free"
                    )
                elif cumulative_interactions > 0:
                    interaction_types[horizon] = "collision"
                elif cumulative_crossings > 0:
                    interaction_types[horizon] = "door_cross"
                else:
                    interaction_types[horizon] = "free"
    if wanted != set(observations):
        raise RuntimeError(f"missing horizons: {wanted - set(observations)}")
    return (
        initial,
        restored,
        observations,
        states,
        interactions,
        interaction_types,
    )


def exact_restore_test(repo, environment, task, state, actions):
    endpoints = []
    images = []
    interactions = []
    for _ in range(3):
        initial, _, _, states, counts, kinds = rollout_branch(
            repo,
            environment,
            task,
            state,
            actions,
            SEED + 9000,
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        interactions.append(
            (counts[max(HORIZONS)], kinds[max(HORIZONS)])
        )
    result = {
        "environment": environment,
        "repeats": 3,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            interactions[0] == item for item in interactions[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(
                np.max(np.abs(endpoints[0] - item))
                for item in endpoints[1:]
            )
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result


def goal_observation(repo, environment, task):
    if environment == "PushT":
        goal = task["goal"]
        state = np.array(
            [80.0, 450.0, goal[0], goal[1], goal[2], 0.0, 0.0]
        )
    else:
        state = np.asarray(task["goal"], dtype=np.float64)
    _, observation, _ = reset_environment(
        repo,
        environment,
        task,
        state,
        SEED + 11000 + task["task_id"],
    )
    return observation


In [ ]:
# Phase A — reconstruct the prior development interventions and retain every model step.
def generate_simulator_truth():
    repo = configure_repo()
    task_payload = []
    split_payload = {
        "protocol": (
            "exploratory task-disjoint training/calibration/development; "
            "the previously inspected final partition is relabeled development_holdout"
        ),
        "environments": {},
    }
    restore_payload = {}
    design_payload = {}

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        truth_dir.mkdir(parents=True, exist_ok=True)
        records = build_state_records(environment)
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        for task in TASKS[environment]:
            task_payload.append(task)

        split_payload["environments"][environment] = {
            name: {
                "task_ids": sorted(
                    task["task_id"]
                    for task in TASKS[environment]
                    if task["split"] == name
                ),
                "state_ids": sorted(
                    record["state_id"]
                    for record in records
                    if record["split"] == name
                ),
            }
            for name in SPLIT_NAMES
        }

        primitive_steps = max(HORIZONS) * FRAMESKIP
        first = records[0]
        first_task = tasks_by_id[first["task_id"]]
        first_actions, labels, selected = candidate_library(
            environment,
            first["state"],
            first_task,
            primitive_steps,
        )
        restore_payload[environment] = exact_restore_test(
            repo,
            environment,
            first_task,
            first["state"],
            first_actions[1],
        )

        state_matrix = []
        task_ids = []
        action_bank = []
        physical_costs = []
        interaction_bank = []
        interaction_type_bank = []
        for record in records:
            state_id = record["state_id"]
            state_path = truth_dir / f"state_{state_id:04d}.npz"
            if state_path.exists():
                log.info("%s simulator resume: keeping %s", environment, state_path.name)
                with np.load(state_path) as shard:
                    state_matrix.append(shard["initial_state"])
                    task_ids.append(int(shard["task_id"]))
                    action_bank.append(shard["selected_actions"])
                    physical_costs.append(shard["physical_cost"])
                    interaction_bank.append(shard["interactions"])
                    interaction_type_bank.append(shard["interaction_types"])
                continue

            task = tasks_by_id[record["task_id"]]
            actions, action_labels, selected_indices = candidate_library(
                environment,
                record["state"],
                task,
                primitive_steps,
            )
            if action_labels != labels:
                raise AssertionError("candidate labels changed across states")
            initials = []
            visuals = []
            proprios = []
            endpoints = []
            all_visuals = []
            all_proprios = []
            all_endpoints = []
            interactions = []
            interaction_types = []
            for branch in actions:
                (
                    initial,
                    _,
                    observations,
                    states,
                    counts,
                    kinds,
                ) = rollout_branch(
                    repo,
                    environment,
                    task,
                    record["state"],
                    branch,
                    record["evaluation_seed"] * 1000 + state_id,
                )
                initials.append(initial["visual"])
                visuals.append(
                    [observations[horizon]["visual"] for horizon in HORIZONS]
                )
                proprios.append(
                    [observations[horizon]["proprio"] for horizon in HORIZONS]
                )
                endpoints.append(
                    [states[horizon] for horizon in HORIZONS]
                )
                all_visuals.append(
                    [observations[step]["visual"] for step in TARGET_STEPS]
                )
                all_proprios.append(
                    [observations[step]["proprio"] for step in TARGET_STEPS]
                )
                all_endpoints.append(
                    [states[step] for step in TARGET_STEPS]
                )
                interactions.append(
                    [counts[horizon] for horizon in HORIZONS]
                )
                interaction_types.append(
                    [kinds[horizon] for horizon in HORIZONS]
                )
            if not all(
                np.array_equal(initials[0], item) for item in initials[1:]
            ):
                raise AssertionError(
                    f"branch initial render mismatch: {environment} state {state_id}"
                )
            endpoint_array = np.asarray(endpoints, dtype=np.float32)
            physical_cost = task_cost(
                environment, endpoint_array, task
            ).astype(np.float32)
            _, initial_observation, _ = reset_environment(
                repo,
                environment,
                task,
                record["state"],
                record["evaluation_seed"] * 1000 + state_id,
            )
            atomic_npz(
                state_path,
                initial_state=np.asarray(record["state"], dtype=np.float64),
                task_id=np.asarray(record["task_id"], dtype=np.int64),
                task_split=np.asarray(record["split"]),
                evaluation_seed=np.asarray(
                    record["evaluation_seed"], dtype=np.int64
                ),
                design_stratum=np.asarray(record["design_stratum"]),
                initial_visual=initials[0],
                initial_proprio=initial_observation["proprio"],
                selected_actions=actions,
                selected_library_indices=selected_indices,
                action_labels=np.asarray(action_labels),
                future_visual=np.asarray(visuals, dtype=np.uint8),
                future_proprio=np.asarray(proprios, dtype=np.float32),
                endpoint_states=endpoint_array,
                all_future_visual=np.asarray(all_visuals, dtype=np.uint8),
                all_future_proprio=np.asarray(all_proprios, dtype=np.float32),
                all_endpoint_states=np.asarray(all_endpoints, dtype=np.float32),
                physical_cost=physical_cost,
                interactions=np.asarray(interactions, dtype=np.int32),
                interaction_types=np.asarray(interaction_types),
            )
            state_matrix.append(record["state"])
            task_ids.append(record["task_id"])
            action_bank.append(actions)
            physical_costs.append(physical_cost)
            interaction_bank.append(interactions)
            interaction_type_bank.append(interaction_types)
            write_json(
                OUT / f"{environment.lower()}_simulator_progress.json",
                {
                    "run_signature": RUN_SIGNATURE,
                    "environment": environment,
                    "completed_states": state_id + 1,
                    "total_states": NUM_STATES,
                    "last_file": state_path.name,
                },
            )
            log.info(
                "%s simulator state %d/%d",
                environment,
                state_id + 1,
                NUM_STATES,
            )

        physical_costs = np.asarray(physical_costs, dtype=np.float64)
        interactions = np.asarray(interaction_bank, dtype=np.int32)
        oracle = np.argmin(physical_costs, axis=1)
        spread = np.max(physical_costs, axis=1) - np.min(
            physical_costs, axis=1
        )
        no_op_regret = physical_costs[:, 0] - np.min(
            physical_costs, axis=1
        )
        left, right = pair_indices(ACTIONS_PER_STATE)
        pair_interactions = (
            (interactions[:, left, :] > 0).astype(int)
            + (interactions[:, right, :] > 0).astype(int)
        )
        environment_design = {
            "selection_protocol": (
                "fixed state/task-relative candidates; future simulator outcomes "
                "are never used for candidate selection"
            ),
            "candidate_labels": labels,
            "no_op_oracle_fraction_by_horizon": np.mean(
                oracle == 0, axis=0
            ).tolist(),
            "no_op_positive_regret_fraction_by_horizon": np.mean(
                no_op_regret > 1e-9, axis=0
            ).tolist(),
            "median_physical_cost_spread_by_horizon": np.median(
                spread, axis=0
            ).tolist(),
            "minimum_physical_cost_spread_by_horizon": np.min(
                spread, axis=0
            ).tolist(),
            "interaction_fraction_by_horizon": np.mean(
                interactions > 0, axis=(0, 1)
            ).tolist(),
            "pair_interaction_counts": {
                label: int(np.sum(pair_interactions == index))
                for index, label in enumerate(["neither", "one", "both"])
            },
        }
        environment_design["validity_thresholds"] = {
            "final_horizon_no_op_oracle_fraction_max": 0.25,
            "final_horizon_no_op_positive_regret_fraction_min": 0.75,
            "final_horizon_median_cost_spread_min": (
                0.08 if environment == "PushT" else 0.05
            ),
            "all_pair_interaction_strata_required": True,
        }
        environment_design["design_valid"] = bool(
            environment_design[
                "no_op_oracle_fraction_by_horizon"
            ][-1]
            < 0.25
            and environment_design[
                "no_op_positive_regret_fraction_by_horizon"
            ][-1]
            > 0.75
            and environment_design[
                "median_physical_cost_spread_by_horizon"
            ][-1]
            > (0.08 if environment == "PushT" else 0.05)
            and all(
                environment_design["pair_interaction_counts"][label] > 0
                for label in ["neither", "one", "both"]
            )
        )
        design_payload[environment] = environment_design
        atomic_npz(
            OUT / f"{environment.lower()}_design.npz",
            states=np.asarray(state_matrix),
            task_ids=np.asarray(task_ids),
            action_bank=np.asarray(action_bank, dtype=np.float32),
            physical_cost=physical_costs.astype(np.float32),
            interactions=interactions,
            interaction_types=np.asarray(interaction_type_bank),
            candidate_labels=np.asarray(labels),
        )

    write_json(OUT / "tasks.json", task_payload)
    write_json(OUT / "split_manifest.json", split_payload)
    write_json(OUT / "restore_test.json", restore_payload)
    write_json(OUT / "candidate_design_summary.json", design_payload)
    return repo


if not PIPELINE_FAILED:
    try:
        REPO = generate_simulator_truth()
    except Exception:
        record_failure("simulator_truth")


In [ ]:
# Phase B — cache the true and predicted transition tokens and audit every
# JEPA-WM AdaLN block.  Intermediates remain outside the downloadable bundle.

TRANSITION_ROOT = INTERMEDIATE / "transitions"
GOAL_ROOT = INTERMEDIATE / "goals"
for path in [TRANSITION_ROOT, GOAL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

AUDIT_LAYERS = [f"block_{index + 1}" for index in range(6)] + [
    "predictor_output"
]


def atomic_npz_uncompressed(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez(temporary, **arrays)
    temporary.replace(path)


def visual_tokens(value):
    """Return [batch, tokens, channels] without spatial pooling."""
    if hasattr(value, "detach"):
        value = value.detach().float().cpu().numpy()
    value = np.asarray(value)
    if value.ndim == 6:
        # [B, T, V, H, W, D] -> last time step.
        value = value[:, -1, 0]
    elif value.ndim == 5:
        # [B, V, H, W, D].
        value = value[:, 0]
    elif value.ndim != 3:
        raise ValueError(f"unexpected visual-token shape {value.shape}")
    return value.reshape(value.shape[0], -1, value.shape[-1])


class CountSketchProjector:
    """Deterministic memory-light projection of flattened patch deltas."""

    def __init__(self, input_dim, output_dim, seed, device="cuda"):
        rng = np.random.default_rng(seed)
        bucket = rng.integers(0, output_dim, size=input_dim, dtype=np.int64)
        sign = rng.choice(np.asarray([-1.0, 1.0], dtype=np.float32), input_dim)
        self.bucket = torch.as_tensor(bucket, device=device, dtype=torch.long)
        self.sign = torch.as_tensor(sign, device=device, dtype=torch.float32)
        counts = np.bincount(bucket, minlength=output_dim).astype(np.float32)
        counts[counts == 0] = 1.0
        self.scale = torch.as_tensor(
            np.sqrt(counts), device=device, dtype=torch.float32
        )
        self.output_dim = int(output_dim)

    def __call__(self, values):
        values = values.float().flatten(1)
        output = torch.zeros(
            values.shape[0],
            self.output_dim,
            device=values.device,
            dtype=torch.float32,
        )
        output.scatter_add_(
            1,
            self.bucket[None].expand(values.shape[0], -1),
            values * self.sign[None],
        )
        return output / self.scale[None]


def model_action_tensor(preprocessor, selected_actions):
    chunks = torch.from_numpy(
        selected_actions.reshape(
            ACTIONS_PER_STATE,
            max(HORIZONS),
            FRAMESKIP,
            2,
        )
    ).float()
    normalized = preprocessor.normalize_actions(chunks)
    return (
        normalized.reshape(ACTIONS_PER_STATE, max(HORIZONS), -1)
        .permute(1, 0, 2)
        .contiguous()
        .cuda()
    )


def validate_jepa_predictor(model, model_name):
    predictor = model.model.predictor
    blocks = list(getattr(predictor, "predictor_blocks", []))
    if len(blocks) != 6:
        raise RuntimeError(
            f"{model_name} expected six AdaLN blocks, found {len(blocks)}"
        )
    if predictor.__class__.__name__ != "VisionTransformerAdaLN":
        raise RuntimeError(
            f"{model_name} is not the expected AdaLN predictor: "
            f"{predictor.__class__.__name__}"
        )
    if not bool(getattr(predictor, "action_encoder_inpred", False)):
        raise RuntimeError(f"{model_name} does not encode actions in predictor")
    return predictor, blocks


def encode_all_true_tokens(model, future_visual, future_proprio):
    encoded = model.encode(
        to_model_observation(future_visual, future_proprio)
    )
    visual = encoded["visual"].detach().float().cpu().numpy()
    # [A, S, 1, H, W, D] -> [A, S, H*W, D]
    return visual[:, :, 0].reshape(
        visual.shape[0], visual.shape[1], -1, visual.shape[-1]
    )


def layer_tokens_from_capture(capture, batch, time_steps, visual_dim):
    value = capture
    if value.ndim != 3:
        raise ValueError(f"unexpected AdaLN block output {tuple(value.shape)}")
    tokens_per_step = value.shape[1] // time_steps
    if tokens_per_step != 16 * 16:
        raise ValueError(
            f"unexpected tokens per step {tokens_per_step}; expected 256"
        )
    return value.view(
        batch, time_steps, tokens_per_step, value.shape[-1]
    )[:, -1, :, :visual_dim]


def base_unroll_with_layer_audit(model, initial_encoded, model_actions):
    predictor, blocks = validate_jepa_predictor(model, "loaded_model")
    visual_dim = int(predictor.predictor_embed_dim)
    projection_input_dim = 16 * 16 * visual_dim
    projectors = [
        CountSketchProjector(
            projection_input_dim,
            AUDIT_PROJECTION_DIM,
            seed,
        )
        for seed in AUDIT_PROJECTION_SEEDS
    ]

    captures = []
    handles = []
    for block in blocks:
        handles.append(
            block.register_forward_hook(
                lambda _module, _inputs, output: captures.append(output)
            )
        )

    try:
        action_batch = model_actions.permute(1, 0, 2).contiguous()
        action_features = model.model.encode_act(action_batch)
        visual_history = initial_encoded["visual"].expand(
            ACTIONS_PER_STATE, *initial_encoded["visual"].shape[1:]
        )
        proprio_history = initial_encoded["proprio"].expand(
            ACTIONS_PER_STATE, *initial_encoded["proprio"].shape[1:]
        )

        base_predictions = []
        base_deltas = []
        base_proprios = []
        audit_steps = []

        for step_index in range(max(HORIZONS)):
            captures.clear()
            action_prefix = action_features[:, : step_index + 1]
            predicted_visual, _, predicted_proprio = model.model.forward_pred(
                visual_history[:, -model.ctxt_window :],
                action_prefix[:, -model.ctxt_window :],
                proprio_history[:, -model.ctxt_window :],
            )
            if len(captures) != len(blocks):
                raise RuntimeError(
                    f"captured {len(captures)} blocks, expected {len(blocks)}"
                )
            next_visual = predicted_visual[:, -1:]
            next_proprio = predicted_proprio[:, -1:]
            current_visual = visual_history[:, -1:]
            next_tokens = next_visual[:, 0, 0].flatten(1, 2)
            current_tokens = current_visual[:, 0, 0].flatten(1, 2)
            base_predictions.append(next_tokens.detach().cpu().numpy())
            base_deltas.append(
                (next_tokens - current_tokens).detach().cpu().numpy()
            )
            base_proprios.append(
                next_proprio[:, 0].detach().float().cpu().numpy()
            )

            time_steps = predicted_visual.shape[1]
            layer_values = [
                layer_tokens_from_capture(
                    captured,
                    ACTIONS_PER_STATE,
                    time_steps,
                    visual_dim,
                )
                for captured in captures
            ]
            layer_values.append(next_tokens)
            per_seed = []
            for projector in projectors:
                per_layer = []
                for value in layer_values:
                    effect = value - value[:1]
                    per_layer.append(projector(effect).detach().cpu().numpy())
                per_seed.append(np.stack(per_layer, axis=0))
            audit_steps.append(np.stack(per_seed, axis=0))

            visual_history = torch.cat(
                [visual_history, next_visual], dim=1
            )
            proprio_history = torch.cat(
                [proprio_history, next_proprio], dim=1
            )

        return {
            "base_prediction": np.stack(base_predictions, axis=1),
            "base_delta": np.stack(base_deltas, axis=1),
            "base_proprio": np.stack(base_proprios, axis=1),
            # [step, projection_seed, layer, action, projection]
            "audit_projection": np.stack(audit_steps, axis=0),
            "normalized_action": action_batch.detach().cpu().numpy(),
        }
    finally:
        for handle in handles:
            handle.remove()


def cache_transition_tokens():
    repo = configure_repo()
    checkpoint_records = []

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            transition_dir = TRANSITION_ROOT / model_name
            transition_dir.mkdir(parents=True, exist_ok=True)
            goal_path = GOAL_ROOT / f"{model_name}.npz"

            torch.cuda.reset_peak_memory_stats()
            gpu_report(f"{model_name}_before_load")
            model, preprocessor = torch.hub.load(
                str(repo),
                model_name,
                source="local",
                pretrained=True,
                device="cuda:0",
                trust_repo=True,
            )
            model.eval()
            predictor, _ = validate_jepa_predictor(model, model_name)
            gpu_report(f"{model_name}_after_load")

            if not goal_path.exists():
                goal_visual = []
                goal_proprio = []
                with torch.inference_mode():
                    for task in TASKS[environment]:
                        observation = goal_observation(repo, environment, task)
                        encoded = model.encode(
                            to_model_observation(
                                observation["visual"],
                                observation["proprio"],
                            )
                        )
                        goal_visual.append(
                            encoded["visual"][0, -1, 0]
                            .flatten(0, 1)
                            .detach()
                            .float()
                            .cpu()
                            .numpy()
                        )
                        goal_proprio.append(
                            encoded["proprio"][0, -1]
                            .detach()
                            .float()
                            .cpu()
                            .numpy()
                        )
                atomic_npz_uncompressed(
                    goal_path,
                    visual=np.asarray(goal_visual, dtype=np.float16),
                    proprio=np.asarray(goal_proprio, dtype=np.float16),
                )

            for state_id in range(NUM_STATES):
                output_path = transition_dir / f"state_{state_id:04d}.npz"
                if output_path.exists():
                    log.info(
                        "%s transition resume: keeping %s",
                        model_name,
                        output_path.name,
                    )
                    continue
                with np.load(
                    truth_dir / f"state_{state_id:04d}.npz"
                ) as truth:
                    initial_visual = truth["initial_visual"]
                    initial_proprio = truth["initial_proprio"]
                    all_future_visual = truth["all_future_visual"]
                    all_future_proprio = truth["all_future_proprio"]
                    selected_actions = truth["selected_actions"]
                    task_id = int(truth["task_id"])

                actions = model_action_tensor(
                    preprocessor, selected_actions
                )
                with torch.inference_mode():
                    initial_encoded = model.encode(
                        to_model_observation(
                            initial_visual,
                            initial_proprio,
                        )
                    )
                    true_tokens = encode_all_true_tokens(
                        model,
                        all_future_visual,
                        all_future_proprio,
                    )
                    cached = base_unroll_with_layer_audit(
                        model,
                        initial_encoded,
                        actions,
                    )

                atomic_npz_uncompressed(
                    output_path,
                    task_id=np.asarray(task_id, dtype=np.int64),
                    true_tokens=true_tokens.astype(np.float16),
                    base_prediction=cached["base_prediction"].astype(
                        np.float16
                    ),
                    base_delta=cached["base_delta"].astype(np.float16),
                    base_proprio=cached["base_proprio"].astype(np.float16),
                    normalized_action=cached["normalized_action"].astype(
                        np.float32
                    ),
                    audit_projection=cached["audit_projection"].astype(
                        np.float16
                    ),
                )
                write_json(
                    OUT / f"{model_name}_transition_progress.json",
                    {
                        "run_signature": RUN_SIGNATURE,
                        "environment": environment,
                        "model": model_name,
                        "completed_states": state_id + 1,
                        "total_states": NUM_STATES,
                        "last_file": output_path.name,
                    },
                )
                log.info(
                    "%s transition state %d/%d",
                    model_name,
                    state_id + 1,
                    NUM_STATES,
                )
                if (state_id + 1) % 12 == 0:
                    gpu_report(
                        f"{model_name}_transition_state_{state_id:04d}"
                    )

            del model, preprocessor, predictor
            gc.collect()
            torch.cuda.empty_cache()
            gpu_report(f"{model_name}_released")

    hf_root = Path(os.environ["HF_HOME"]) / "hub"
    torch_root = Path(os.environ["TORCH_HOME"])
    for root in [hf_root, torch_root]:
        if root.exists():
            for path in root.rglob("*"):
                if (
                    path.is_file()
                    and path.stat().st_size > 20_000_000
                    and path.suffix
                    in {".tar", ".pth", ".pt", ".bin", ".safetensors"}
                ):
                    checkpoint_records.append(
                        {
                            "path": str(path),
                            "size_bytes": path.stat().st_size,
                            "sha256": sha256_file(path),
                        }
                    )
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "models": MODEL_NAME,
            "repository": "facebook/jepa-wms",
            "repository_commit": REPO_COMMIT,
            "predictor_requirement": "six-block VisionTransformerAdaLN",
            "cached_files": checkpoint_records,
        },
    )


if not PIPELINE_FAILED:
    try:
        cache_transition_tokens()
    except Exception:
        record_failure("transition_token_cache_and_layer_audit")


In [ ]:
# Phase C — layerwise audit and offline residual-adapter development.

ADAPTER_DIR = OUT / "trained_recurrent_adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

METHOD_WEIGHTS = {
    "absolute_residual": {
        "counterfactual": 0.0,
        "independent": 0.0,
    },
    "independent_delta_control": {
        "counterfactual": 0.0,
        "independent": COUNTERFACTUAL_WEIGHT,
    },
    "counterfactual_recurrent": {
        "counterfactual": COUNTERFACTUAL_WEIGHT,
        "independent": 0.0,
    },
}


def stable_seed(*items):
    digest = hashlib.sha256()
    for item in items:
        digest.update(str(item).encode())
        digest.update(b"\0")
    return int.from_bytes(digest.digest()[:8], "little") % (2**31 - 1)


def canonical_split_name(split_name):
    # Backward-compatible with the first Stage 7 cache, whose held-out
    # partition was numerically correct but retained the old Stage 5 label.
    value = str(split_name)
    return DEVELOPMENT_SPLIT if value == "final_test" else value


def split_state_ids(environment, split_name):
    truth_dir = TRUTH_ROOT / environment.lower()
    output = []
    for state_id in range(NUM_STATES):
        with np.load(truth_dir / f"state_{state_id:04d}.npz") as truth:
            if canonical_split_name(truth["task_split"]) == split_name:
                output.append(state_id)
    if not output:
        available = sorted(
            {
                canonical_split_name(
                    np.load(
                        truth_dir / f"state_{state_id:04d}.npz"
                    )["task_split"]
                )
                for state_id in range(NUM_STATES)
            }
        )
        raise RuntimeError(
            f"{environment} split {split_name!r} has no states; "
            f"available splits: {available}"
        )
    return output


def load_transition_states(model_name, state_ids):
    transition_dir = TRANSITION_ROOT / model_name
    values = {
        "true_tokens": [],
        "base_prediction": [],
        "base_delta": [],
        "base_proprio": [],
        "normalized_action": [],
        "state_id": [],
        "task_id": [],
    }
    for state_id in state_ids:
        with np.load(
            transition_dir / f"state_{state_id:04d}.npz"
        ) as shard:
            for key in [
                "true_tokens",
                "base_prediction",
                "base_delta",
                "base_proprio",
                "normalized_action",
            ]:
                values[key].append(shard[key])
            values["task_id"].append(int(shard["task_id"]))
            values["state_id"].append(state_id)
    return {
        key: np.asarray(value)
        for key, value in values.items()
    }


class TokenResidualAdapter(torch.nn.Module):
    """Small shared patch residual applied at every recurrent step."""

    def __init__(
        self,
        token_dim,
        action_dim,
        hidden_dim,
        steps,
        tokens_per_frame=256,
    ):
        super().__init__()
        self.delta_norm = torch.nn.LayerNorm(
            token_dim, elementwise_affine=False
        )
        self.action_projection = torch.nn.Sequential(
            torch.nn.Linear(action_dim, token_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(token_dim, token_dim),
        )
        self.step_embedding = torch.nn.Embedding(steps, token_dim)
        self.position_embedding = torch.nn.Parameter(
            torch.zeros(tokens_per_frame, token_dim)
        )
        self.residual = torch.nn.Sequential(
            torch.nn.Linear(token_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, token_dim),
        )
        torch.nn.init.normal_(self.position_embedding, std=0.01)
        torch.nn.init.zeros_(self.residual[-1].weight)
        torch.nn.init.zeros_(self.residual[-1].bias)

    def forward(self, base_prediction, base_delta, action, step_index):
        action_value = self.action_projection(action)[:, None, :]
        step_value = self.step_embedding(step_index)[:, None, :]
        hidden = (
            self.delta_norm(base_delta)
            + action_value
            + step_value
            + self.position_embedding[None]
        )
        return base_prediction + self.residual(hidden)


def dynamic_token_weights(true_tokens):
    effect = true_tokens - true_tokens[:, :1]
    magnitude = torch.sqrt(
        torch.mean(effect.float() ** 2, dim=(1, 4)) + 1e-8
    )
    scale = torch.mean(magnitude, dim=1, keepdim=True).clamp_min(1e-6)
    normalized = magnitude / scale
    return torch.clamp(0.25 + normalized, min=0.25, max=4.0)


def weighted_token_huber(prediction, target, token_weight):
    value = torch_functional.smooth_l1_loss(
        prediction.float(), target.float(), reduction="none"
    ).mean(dim=-1)
    return torch.mean(value * token_weight[:, None, :, :])


def recurrent_adapter_objective(method, corrected, target):
    if method not in METHOD_WEIGHTS:
        raise KeyError(method)
    weight = dynamic_token_weights(target)
    absolute = weighted_token_huber(corrected, target, weight)
    predicted_effect = corrected - corrected[:, :1]
    true_effect = target - target[:, :1]
    counterfactual = weighted_token_huber(
        predicted_effect, true_effect, weight
    )
    partner_noop = torch.roll(target[:, :1], shifts=1, dims=0)
    independent_target = target - partner_noop
    independent = weighted_token_huber(
        predicted_effect, independent_target, weight
    )
    weights = METHOD_WEIGHTS[method]
    total = (
        absolute
        + weights["counterfactual"] * counterfactual
        + weights["independent"] * independent
    )
    return total, {
        "absolute_loss": absolute,
        "counterfactual_delta_loss": counterfactual,
        "independent_delta_loss": independent,
    }


def corrected_cached_tokens(adapter, batch, device="cuda"):
    base_prediction = torch.as_tensor(
        batch["base_prediction"], device=device, dtype=torch.float32
    )
    base_delta = torch.as_tensor(
        batch["base_delta"], device=device, dtype=torch.float32
    )
    action = torch.as_tensor(
        batch["normalized_action"], device=device, dtype=torch.float32
    )
    state_count, action_count, steps, token_count, token_dim = (
        base_prediction.shape
    )
    prediction = base_prediction.permute(0, 2, 1, 3, 4).reshape(
        state_count * steps * action_count, token_count, token_dim
    )
    delta = base_delta.permute(0, 2, 1, 3, 4).reshape_as(prediction)
    action_value = action.permute(0, 2, 1, 3).reshape(
        state_count * steps * action_count, -1
    )
    step_index = (
        torch.arange(steps, device=device)
        .view(1, steps, 1)
        .expand(state_count, steps, action_count)
        .reshape(-1)
    )
    corrected = adapter(prediction, delta, action_value, step_index)
    return corrected.view(
        state_count, steps, action_count, token_count, token_dim
    ).permute(0, 2, 1, 3, 4)


def official_visual_cost(tokens, goal_tokens):
    difference = tokens.float() - goal_tokens.float()
    return torch.mean(difference**2, dim=(-1, -2))


def cached_calibration_score(
    environment,
    model_name,
    adapter,
    calibration,
):
    adapter.eval()
    with np.load(GOAL_ROOT / f"{model_name}.npz") as goal_file:
        goal_visual = goal_file["visual"].astype(np.float32)
        goal_proprio = goal_file["proprio"].astype(np.float32)
    with torch.inference_mode():
        corrected = corrected_cached_tokens(adapter, calibration)
        target = torch.as_tensor(
            calibration["true_tokens"],
            device="cuda",
            dtype=torch.float32,
        )
        absolute_rmse = torch.sqrt(
            torch.mean((corrected - target) ** 2)
        ).item()
        predicted_effect = corrected - corrected[:, :1]
        true_effect = target - target[:, :1]
        paired_rmse = torch.sqrt(
            torch.mean((predicted_effect - true_effect) ** 2)
        ).item()
        effect_scale = torch.sqrt(torch.mean(true_effect**2)).item()
        normalized_paired = paired_rmse / max(effect_scale, 1e-8)

        corrected_np = corrected.detach().cpu().numpy()
    regrets = []
    weighted = []
    truth_dir = TRUTH_ROOT / environment.lower()
    horizon_positions = [step - 1 for step in HORIZONS]
    for local_index, state_id in enumerate(calibration["state_id"]):
        task_id = int(calibration["task_id"][local_index])
        visual_goal = goal_visual[task_id]
        proprio_goal = goal_proprio[task_id]
        predicted_cost = np.mean(
            (
                corrected_np[local_index, :, horizon_positions]
                - visual_goal[None, None]
            )
            ** 2,
            axis=(-1, -2),
        )
        predicted_cost += 0.1 * np.mean(
            (
                calibration["base_proprio"][
                    local_index, :, horizon_positions
                ]
                - proprio_goal[None, None]
            )
            ** 2,
            axis=(-1, -2),
        )
        with np.load(
            truth_dir / f"state_{int(state_id):04d}.npz"
        ) as truth:
            physical = truth["physical_cost"].astype(np.float64)
        for horizon_index in range(len(HORIZONS)):
            metrics = ranking_metrics(
                physical[:, horizon_index],
                predicted_cost[:, horizon_index],
            )
            regrets.append(metrics["normalized_regret"])
            weighted.append(metrics["weighted_pairwise_accuracy"])
    regret = float(np.mean(regrets))
    accuracy = float(np.nanmean(weighted))
    score = (
        normalized_paired
        + regret
        + (1.0 - accuracy)
        + 0.25 * absolute_rmse
    )
    return {
        "selection_score": float(score),
        "absolute_latent_rmse": float(absolute_rmse),
        "normalized_paired_latent_rmse": float(normalized_paired),
        "normalized_regret": regret,
        "weighted_pairwise_accuracy": accuracy,
    }


def adapter_state_hash(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.state_dict().items()):
        digest.update(key.encode())
        digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


def train_recurrent_adapter(
    environment,
    model_name,
    adapter_seed,
    method,
    train,
    calibration,
):
    random.seed(stable_seed("adapter", model_name, adapter_seed))
    np.random.seed(stable_seed("adapter_np", model_name, adapter_seed))
    torch.manual_seed(stable_seed("adapter_torch", model_name, adapter_seed))
    torch.cuda.manual_seed_all(
        stable_seed("adapter_cuda", model_name, adapter_seed)
    )
    token_dim = int(train["base_prediction"].shape[-1])
    action_dim = int(train["normalized_action"].shape[-1])
    adapter = TokenResidualAdapter(
        token_dim=token_dim,
        action_dim=action_dim,
        hidden_dim=ADAPTER_HIDDEN_DIM,
        steps=max(HORIZONS),
    ).cuda()
    initial_hash = adapter_state_hash(adapter)
    optimizer = torch.optim.AdamW(
        adapter.parameters(),
        lr=TRAINING_LR,
        weight_decay=TRAINING_WEIGHT_DECAY,
    )
    checkpoint = (
        ADAPTER_DIR
        / f"{model_name}_seed{adapter_seed}_{method}.pt"
    )
    history = []
    candidates = []
    generator = np.random.default_rng(
        stable_seed("adapter_order", model_name, adapter_seed)
    )

    for epoch in range(1, TRAINING_EPOCHS + 1):
        adapter.train()
        order = generator.permutation(len(train["state_id"]))
        component_values = {
            "loss": [],
            "absolute_loss": [],
            "counterfactual_delta_loss": [],
            "independent_delta_loss": [],
        }
        for start in range(0, len(order), TRAINING_BATCH_STATES):
            indices = order[start : start + TRAINING_BATCH_STATES]
            batch = {
                key: value[indices]
                for key, value in train.items()
                if key
                in {
                    "base_prediction",
                    "base_delta",
                    "normalized_action",
                    "true_tokens",
                }
            }
            optimizer.zero_grad(set_to_none=True)
            corrected = corrected_cached_tokens(adapter, batch)
            target = torch.as_tensor(
                batch["true_tokens"],
                device="cuda",
                dtype=torch.float32,
            )
            total, components = recurrent_adapter_objective(
                method, corrected, target
            )
            if not torch.isfinite(total):
                raise FloatingPointError(
                    f"nonfinite loss {model_name} {method} epoch {epoch}"
                )
            total.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            optimizer.step()
            component_values["loss"].append(float(total.detach().cpu()))
            for key, value in components.items():
                component_values[key].append(
                    float(value.detach().cpu())
                )

        row = {
            "environment": environment,
            "model": model_name,
            "adapter_seed": int(adapter_seed),
            "method": method,
            "epoch": int(epoch),
            **{
                key: float(np.mean(value))
                for key, value in component_values.items()
            },
        }
        history.append(row)
        if epoch in SELECTION_EPOCHS:
            metrics = cached_calibration_score(
                environment,
                model_name,
                adapter,
                calibration,
            )
            candidates.append(
                {
                    **row,
                    **metrics,
                    "state_dict": {
                        key: value.detach().cpu().clone()
                        for key, value in adapter.state_dict().items()
                    },
                }
            )
        log.info(
            "%s %s seed=%d epoch=%d loss=%.6f",
            model_name,
            method,
            adapter_seed,
            epoch,
            row["loss"],
        )

    best = min(candidates, key=lambda item: item["selection_score"])
    adapter.load_state_dict(best["state_dict"])
    torch.save(
        {
            "state_dict": adapter.state_dict(),
            "environment": environment,
            "model": model_name,
            "adapter_seed": int(adapter_seed),
            "method": method,
            "selected_epoch": int(best["epoch"]),
            "selection_score": float(best["selection_score"]),
            "initial_parameter_sha256": initial_hash,
            "implementation_id": ADAPTER_IMPLEMENTATION_ID,
            "token_dim": token_dim,
            "action_dim": action_dim,
            "hidden_dim": ADAPTER_HIDDEN_DIM,
        },
        checkpoint,
    )
    selection_rows = []
    for candidate in candidates:
        selection_rows.append(
            {
                key: value
                for key, value in candidate.items()
                if key != "state_dict"
            }
            | {
                "selected": bool(candidate["epoch"] == best["epoch"])
            }
        )
    return adapter, history, selection_rows, checkpoint, initial_hash


def fit_ridge_projection(x_train, y_train, x_calibration, y_calibration):
    x_mean = np.mean(x_train, axis=0, dtype=np.float64)
    x_scale = np.std(x_train, axis=0, dtype=np.float64)
    x_scale[x_scale < 1e-7] = 1.0
    y_mean = np.mean(y_train, axis=0, dtype=np.float64)
    train_x = (
        np.asarray(x_train, dtype=np.float64) - x_mean
    ) / x_scale
    calibration_x = (
        np.asarray(x_calibration, dtype=np.float64) - x_mean
    ) / x_scale
    train_x = np.column_stack([np.ones(len(train_x)), train_x])
    calibration_x = np.column_stack(
        [np.ones(len(calibration_x)), calibration_x]
    )
    gram = train_x.T @ train_x
    cross = train_x.T @ np.asarray(y_train, dtype=np.float64)
    best = None
    for ridge in RIDGE_LAMBDAS:
        penalty = np.eye(gram.shape[0]) * ridge
        penalty[0, 0] = 0.0
        coefficient = np.linalg.solve(gram + penalty, cross)
        prediction = calibration_x @ coefficient
        loss = float(np.mean((prediction - y_calibration) ** 2))
        candidate = {
            "ridge": float(ridge),
            "calibration_mse": loss,
            "x_mean": x_mean,
            "x_scale": x_scale,
            "y_mean": y_mean,
            "coefficient": coefficient,
        }
        if best is None or loss < best["calibration_mse"]:
            best = candidate
    return best


def ridge_prediction(probe, values):
    values = (
        np.asarray(values, dtype=np.float64) - probe["x_mean"]
    ) / probe["x_scale"]
    values = np.column_stack([np.ones(len(values)), values])
    return values @ probe["coefficient"]


def audit_arrays(environment, model_name, split_name):
    state_ids = split_state_ids(environment, split_name)
    transition_dir = TRANSITION_ROOT / model_name
    truth_dir = TRUTH_ROOT / environment.lower()
    features = []
    effects = []
    identifiers = []
    for state_id in state_ids:
        with np.load(
            transition_dir / f"state_{state_id:04d}.npz"
        ) as transition, np.load(
            truth_dir / f"state_{state_id:04d}.npz"
        ) as truth:
            projection = transition["audit_projection"].astype(np.float32)
            # [step, projection_seed, layer, action, dim]
            endpoint = pose_target(
                environment, truth["all_endpoint_states"]
            ).astype(np.float32)
            endpoint_effect = endpoint - endpoint[:1]
            features.append(projection)
            effects.append(endpoint_effect)
            identifiers.append(state_id)
    return {
        "features": np.asarray(features),
        "effects": np.asarray(effects),
        "state_id": np.asarray(identifiers, dtype=np.int64),
    }


def run_layerwise_audit():
    rows = []
    gate_payload = {
        "evidence_status": EVIDENCE_STATUS,
        "criterion": (
            "positive development effect R2 at any internal layer and "
            "at least one horizon in each environment"
        ),
        "environments": {},
    }
    for environment in ENVIRONMENT:
        environment_best = -np.inf
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            train = audit_arrays(environment, model_name, "probe_train")
            calibration = audit_arrays(
                environment, model_name, "probe_calibration"
            )
            development = audit_arrays(
                environment, model_name, DEVELOPMENT_SPLIT
            )
            for projection_index, projection_seed in enumerate(
                AUDIT_PROJECTION_SEEDS
            ):
                for layer_index, layer_name in enumerate(AUDIT_LAYERS):
                    for step_index in range(max(HORIZONS)):
                        train_x = train["features"][
                            :, step_index, projection_index, layer_index
                        ][:, 1:].reshape(
                            -1, AUDIT_PROJECTION_DIM
                        )
                        calibration_x = calibration["features"][
                            :, step_index, projection_index, layer_index
                        ][:, 1:].reshape(
                            -1, AUDIT_PROJECTION_DIM
                        )
                        development_x = development["features"][
                            :, step_index, projection_index, layer_index
                        ][:, 1:].reshape(
                            -1, AUDIT_PROJECTION_DIM
                        )
                        train_y = train["effects"][
                            :, 1:, step_index
                        ].reshape(-1, train["effects"].shape[-1])
                        calibration_y = calibration["effects"][
                            :, 1:, step_index
                        ].reshape(
                            -1, calibration["effects"].shape[-1]
                        )
                        development_y = development["effects"][
                            :, 1:, step_index
                        ].reshape(
                            -1, development["effects"].shape[-1]
                        )
                        probe = fit_ridge_projection(
                            train_x,
                            train_y,
                            calibration_x,
                            calibration_y,
                        )
                        prediction = ridge_prediction(
                            probe, development_x
                        )
                        mse = float(
                            np.mean((prediction - development_y) ** 2)
                        )
                        baseline_mse = float(
                            np.mean(
                                (
                                    development_y
                                    - probe["y_mean"][None]
                                )
                                ** 2
                            )
                        )
                        r2 = (
                            1.0 - mse / baseline_mse
                            if baseline_mse > 1e-12
                            else float("nan")
                        )
                        normalized_rmse = (
                            math.sqrt(mse / baseline_mse)
                            if baseline_mse > 1e-12
                            else float("nan")
                        )
                        environment_best = max(environment_best, r2)
                        rows.append(
                            {
                                "environment": environment,
                                "model": model_name,
                                "projection_seed": int(projection_seed),
                                "layer": layer_name,
                                "layer_index": int(layer_index + 1),
                                "step": int(step_index + 1),
                                "selected_ridge": probe["ridge"],
                                "calibration_mse": probe[
                                    "calibration_mse"
                                ],
                                "development_effect_mse": mse,
                                "development_effect_r2": float(r2),
                                "development_effect_normalized_rmse": float(
                                    normalized_rmse
                                ),
                                "development_states": int(
                                    len(development["state_id"])
                                ),
                            }
                        )
        gate_payload["environments"][environment] = {
            "best_development_effect_r2": float(environment_best),
            "repairable_signal": bool(environment_best > 0.0),
        }
    gate_payload["all_environments_repairable"] = bool(
        all(
            value["repairable_signal"]
            for value in gate_payload["environments"].values()
        )
    )
    write_csv(OUT / "layerwise_audit.csv", rows)
    write_json(OUT / "layerwise_audit_gate.json", gate_payload)
    return rows, gate_payload


def develop_recurrent_adapters():
    history_rows = []
    selection_rows = []
    manifest_records = []
    for environment in ENVIRONMENT:
        train_ids = split_state_ids(environment, "probe_train")
        calibration_ids = split_state_ids(
            environment, "probe_calibration"
        )
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            log.info("%s loading transition tensors for training", model_name)
            train = load_transition_states(model_name, train_ids)
            calibration = load_transition_states(
                model_name, calibration_ids
            )
            for adapter_seed in ADAPTER_SEEDS:
                initial_hashes = {}
                for method in ADAPTER_METHODS:
                    (
                        _adapter,
                        history,
                        selections,
                        checkpoint,
                        initial_hash,
                    ) = train_recurrent_adapter(
                        environment,
                        model_name,
                        adapter_seed,
                        method,
                        train,
                        calibration,
                    )
                    history_rows.extend(history)
                    selection_rows.extend(selections)
                    initial_hashes[method] = initial_hash
                    selected = next(
                        row for row in selections if row["selected"]
                    )
                    manifest_records.append(
                        {
                            "environment": environment,
                            "model": model_name,
                            "adapter_seed": int(adapter_seed),
                            "method": method,
                            "initial_parameter_sha256": initial_hash,
                            "selected_epoch": int(selected["epoch"]),
                            "selected_calibration_score": float(
                                selected["selection_score"]
                            ),
                            "checkpoint": str(
                                checkpoint.relative_to(OUT)
                            ),
                            "checkpoint_sha256": sha256_file(checkpoint),
                        }
                    )
                    del _adapter
                    gc.collect()
                    torch.cuda.empty_cache()
                if len(set(initial_hashes.values())) != 1:
                    raise AssertionError(
                        f"method initialization mismatch: {initial_hashes}"
                    )
            del train, calibration
            gc.collect()

    write_csv(OUT / "adapter_training_history.csv", history_rows)
    write_csv(OUT / "checkpoint_selection.csv", selection_rows)
    write_json(
        OUT / "recurrent_adapter_manifest.json",
        {
            "evidence_status": EVIDENCE_STATUS,
            "world_model_frozen": True,
            "adapter_location": (
                "shared token residual inserted after every forward_pred step "
                "and before the prediction is recycled as context"
            ),
            "ranking_gradient_into_dynamics": False,
            "spatial_pooling_before_adapter": False,
            "methods": ADAPTER_METHODS,
            "method_weights": METHOD_WEIGHTS,
            "training_partition": "probe_train",
            "calibration_used_for_checkpoint_selection": True,
            "development_holdout_used_for_selection": False,
            "records": manifest_records,
        },
    )
    return history_rows, selection_rows


if not PIPELINE_FAILED:
    try:
        LAYER_AUDIT_ROWS, LAYER_AUDIT_GATE = run_layerwise_audit()
        TRAINING_HISTORY_ROWS, CHECKPOINT_SELECTION_ROWS = (
            develop_recurrent_adapters()
        )
    except Exception:
        record_failure("layer_audit_and_recurrent_adapter_development")


In [ ]:
# Phase D — insert selected adapters into the actual autoregressive inference
# loop and evaluate the resulting action ranking.

EVALUATION_METHODS = ["base_world_model"] + ADAPTER_METHODS
LOWER_IS_BETTER = {
    "ordinary_latent_rmse",
    "normalized_paired_latent_rmse",
    "normalized_regret",
    "normalized_margin_rmse",
}


def load_selected_adapter(model_name, adapter_seed, method):
    checkpoint = (
        ADAPTER_DIR
        / f"{model_name}_seed{adapter_seed}_{method}.pt"
    )
    payload = torch.load(
        checkpoint, map_location="cpu", weights_only=False
    )
    if payload["implementation_id"] != ADAPTER_IMPLEMENTATION_ID:
        raise RuntimeError(f"incompatible adapter {checkpoint}")
    adapter = TokenResidualAdapter(
        token_dim=int(payload["token_dim"]),
        action_dim=int(payload["action_dim"]),
        hidden_dim=int(payload["hidden_dim"]),
        steps=max(HORIZONS),
    ).cuda()
    adapter.load_state_dict(payload["state_dict"])
    adapter.eval()
    return adapter, payload


def recurrent_unroll_with_adapter(
    model,
    initial_encoded,
    model_actions,
    adapter=None,
):
    action_batch = model_actions.permute(1, 0, 2).contiguous()
    action_features = model.model.encode_act(action_batch)
    visual_history = initial_encoded["visual"].expand(
        ACTIONS_PER_STATE, *initial_encoded["visual"].shape[1:]
    )
    proprio_history = initial_encoded["proprio"].expand(
        ACTIONS_PER_STATE, *initial_encoded["proprio"].shape[1:]
    )
    output_visual = []
    output_proprio = []
    for step_index in range(max(HORIZONS)):
        predicted_visual, _, predicted_proprio = model.model.forward_pred(
            visual_history[:, -model.ctxt_window :],
            action_features[
                :, : step_index + 1
            ][:, -model.ctxt_window :],
            proprio_history[:, -model.ctxt_window :],
        )
        next_visual = predicted_visual[:, -1:]
        next_proprio = predicted_proprio[:, -1:]
        if adapter is not None:
            current = visual_history[:, -1, 0].flatten(1, 2)
            base = next_visual[:, 0, 0].flatten(1, 2)
            action = action_batch[:, step_index]
            step = torch.full(
                (ACTIONS_PER_STATE,),
                step_index,
                device=base.device,
                dtype=torch.long,
            )
            corrected = adapter(base, base - current, action, step)
            next_visual = corrected.view(
                ACTIONS_PER_STATE,
                1,
                1,
                16,
                16,
                corrected.shape[-1],
            )
        output_visual.append(
            next_visual[:, 0, 0].flatten(1, 2)
        )
        output_proprio.append(next_proprio[:, 0])
        visual_history = torch.cat(
            [visual_history, next_visual], dim=1
        )
        proprio_history = torch.cat(
            [proprio_history, next_proprio], dim=1
        )
    return (
        torch.stack(output_visual, dim=1),
        torch.stack(output_proprio, dim=1),
    )


def paired_latent_metrics(target, prediction):
    target = np.asarray(target, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    ordinary = float(np.sqrt(np.mean((prediction - target) ** 2)))
    target_effect = target - target[:1]
    predicted_effect = prediction - prediction[:1]
    paired = float(
        np.sqrt(np.mean((predicted_effect - target_effect) ** 2))
    )
    scale = float(np.sqrt(np.mean(target_effect**2)))
    normalized = paired / max(scale, 1e-12)
    return ordinary, normalized


def evaluate_recurrent_methods():
    repo = configure_repo()
    unit_rows = []
    action_rows = []
    evaluation_manifest = []
    selected_steps = [step - 1 for step in HORIZONS]

    for environment in ENVIRONMENT:
        truth_dir = TRUTH_ROOT / environment.lower()
        tasks_by_id = {
            item["task_id"]: item for item in TASKS[environment]
        }
        # Calibration is used only to choose a checkpoint from the cached
        # one-step transition objectives.  The expensive autoregressive
        # comparison is then run once on the development holdout.
        evaluation_ids = split_state_ids(
            environment, DEVELOPMENT_SPLIT
        )
        for model_name in MODEL_BY_ENVIRONMENT[environment]:
            model, preprocessor = torch.hub.load(
                str(repo),
                model_name,
                source="local",
                pretrained=True,
                device="cuda:0",
                trust_repo=True,
            )
            model.eval()
            with np.load(
                GOAL_ROOT / f"{model_name}.npz"
            ) as goals:
                goal_visual = goals["visual"].astype(np.float32)
                goal_proprio = goals["proprio"].astype(np.float32)

            method_adapters = {
                "base_world_model": [(None, None)]
            }
            for method in ADAPTER_METHODS:
                method_adapters[method] = [
                    load_selected_adapter(
                        model_name, adapter_seed, method
                    )
                    for adapter_seed in ADAPTER_SEEDS
                ]

            for state_index, state_id in enumerate(evaluation_ids):
                with np.load(
                    truth_dir / f"state_{state_id:04d}.npz"
                ) as truth:
                    split_name = canonical_split_name(
                        truth["task_split"]
                    )
                    task_id = int(truth["task_id"])
                    initial_visual = truth["initial_visual"]
                    initial_proprio = truth["initial_proprio"]
                    selected_actions = truth["selected_actions"]
                    physical_cost = truth["physical_cost"].astype(
                        np.float64
                    )
                with np.load(
                    TRANSITION_ROOT
                    / model_name
                    / f"state_{state_id:04d}.npz"
                ) as cached:
                    true_tokens = cached["true_tokens"].astype(
                        np.float32
                    )
                actions = model_action_tensor(
                    preprocessor, selected_actions
                )
                with torch.inference_mode():
                    initial_encoded = model.encode(
                        to_model_observation(
                            initial_visual,
                            initial_proprio,
                        )
                    )
                    for method, adapters in method_adapters.items():
                        for adapter_position, (
                            adapter,
                            adapter_payload,
                        ) in enumerate(adapters):
                            adapter_seed = (
                                0
                                if adapter_payload is None
                                else int(
                                    adapter_payload["adapter_seed"]
                                )
                            )
                            predicted_visual, predicted_proprio = (
                                recurrent_unroll_with_adapter(
                                    model,
                                    initial_encoded,
                                    actions,
                                    adapter=adapter,
                                )
                            )
                            predicted_visual_np = (
                                predicted_visual.detach()
                                .float()
                                .cpu()
                                .numpy()
                            )
                            visual_goal = torch.as_tensor(
                                goal_visual[task_id],
                                device="cuda",
                                dtype=torch.float32,
                            )
                            proprio_goal = torch.as_tensor(
                                goal_proprio[task_id],
                                device="cuda",
                                dtype=torch.float32,
                            )
                            predicted_cost = (
                                official_visual_cost(
                                    predicted_visual, visual_goal
                                )
                                + 0.1
                                * torch.mean(
                                    (
                                        predicted_proprio.float()
                                        - proprio_goal.float()
                                    )
                                    ** 2,
                                    dim=tuple(
                                        range(
                                            2,
                                            predicted_proprio.ndim,
                                        )
                                    ),
                                )
                            ).detach().cpu().numpy()

                            for horizon_position, horizon in zip(
                                selected_steps, HORIZONS
                            ):
                                ordinary, paired = (
                                    paired_latent_metrics(
                                        true_tokens[
                                            :, horizon_position
                                        ],
                                        predicted_visual_np[
                                            :, horizon_position
                                        ],
                                    )
                                )
                                ranking = ranking_metrics(
                                    physical_cost[
                                        :, HORIZONS.index(horizon)
                                    ],
                                    predicted_cost[
                                        :, horizon_position
                                    ],
                                )
                                unit_rows.append(
                                    {
                                        "environment": environment,
                                        "state_id": int(state_id),
                                        "task_id": int(task_id),
                                        "split": split_name,
                                        "evaluation_seed": int(
                                            truth_evaluation_seed(
                                                truth_dir, state_id
                                            )
                                        ),
                                        "model": model_name,
                                        "model_family": "JEPA-WM",
                                        "adapter_seed": int(
                                            adapter_seed
                                        ),
                                        "method": method,
                                        "horizon": int(horizon),
                                        "ordinary_latent_rmse": ordinary,
                                        "normalized_paired_latent_rmse": paired,
                                        "normalized_regret": ranking[
                                            "normalized_regret"
                                        ],
                                        "weighted_pairwise_accuracy": ranking[
                                            "weighted_pairwise_accuracy"
                                        ],
                                        "top1_correct": ranking[
                                            "top1_correct"
                                        ],
                                        "normalized_margin_rmse": ranking[
                                            "normalized_margin_rmse"
                                        ],
                                        "selected_action": ranking[
                                            "selected_action"
                                        ],
                                        "oracle_action": ranking[
                                            "oracle_action"
                                        ],
                                    }
                                )
                                for action_index in range(
                                    ACTIONS_PER_STATE
                                ):
                                    action_rows.append(
                                        {
                                            "environment": environment,
                                            "state_id": int(state_id),
                                            "task_id": int(task_id),
                                            "split": split_name,
                                            "model": model_name,
                                            "adapter_seed": int(
                                                adapter_seed
                                            ),
                                            "method": method,
                                            "horizon": int(horizon),
                                            "action": int(action_index),
                                            "true_physical_cost": float(
                                                physical_cost[
                                                    action_index,
                                                    HORIZONS.index(
                                                        horizon
                                                    ),
                                                ]
                                            ),
                                            "predicted_latent_cost": float(
                                                predicted_cost[
                                                    action_index,
                                                    horizon_position,
                                                ]
                                            ),
                                        }
                                    )
                if (state_index + 1) % 12 == 0:
                    log.info(
                        "%s recurrent evaluation %d/%d",
                        model_name,
                        state_index + 1,
                        len(evaluation_ids),
                    )

            for method, adapters in method_adapters.items():
                for adapter, payload in adapters:
                    evaluation_manifest.append(
                        {
                            "environment": environment,
                            "model": model_name,
                            "method": method,
                            "adapter_seed": (
                                0
                                if payload is None
                                else int(payload["adapter_seed"])
                            ),
                            "selected_epoch": (
                                None
                                if payload is None
                                else int(payload["selected_epoch"])
                            ),
                        }
                    )
                    if adapter is not None:
                        del adapter
            del model, preprocessor, method_adapters
            gc.collect()
            torch.cuda.empty_cache()

    write_csv(OUT / "unit_metrics.csv", unit_rows)
    write_csv(OUT / "action_predictions.csv", action_rows)
    write_json(
        OUT / "recurrent_evaluation_manifest.json",
        {
            "evidence_status": EVIDENCE_STATUS,
            "recurrent_insertion_used": True,
            "development_holdout_used_for_selection": False,
            "records": evaluation_manifest,
        },
    )
    return unit_rows, action_rows


def truth_evaluation_seed(truth_dir, state_id):
    with np.load(
        truth_dir / f"state_{int(state_id):04d}.npz"
    ) as truth:
        return int(truth["evaluation_seed"])


def development_summary(unit_rows):
    rows = [
        row
        for row in unit_rows
        if row["split"] == DEVELOPMENT_SPLIT
    ]
    output = []
    metrics = [
        "ordinary_latent_rmse",
        "normalized_paired_latent_rmse",
        "normalized_regret",
        "weighted_pairwise_accuracy",
        "top1_correct",
        "normalized_margin_rmse",
    ]
    for environment in ENVIRONMENT:
        for method in EVALUATION_METHODS:
            selected = [
                row
                for row in rows
                if row["environment"] == environment
                and row["method"] == method
            ]
            if not selected:
                continue
            output.append(
                {
                    "environment": environment,
                    "method": method,
                    "n_rows": len(selected),
                    "n_state_clusters": len(
                        {row["state_id"] for row in selected}
                    ),
                    **{
                        metric: float(
                            np.nanmean(
                                [row[metric] for row in selected]
                            )
                        )
                        for metric in metrics
                    },
                }
            )
    return output


def state_metric_map(unit_rows, environment, method, metric):
    grouped = {}
    for row in unit_rows:
        if (
            row["split"] == DEVELOPMENT_SPLIT
            and row["environment"] == environment
            and row["method"] == method
            and np.isfinite(row[metric])
        ):
            grouped.setdefault(int(row["state_id"]), []).append(
                float(row[metric])
            )
    return {
        state_id: float(np.mean(values))
        for state_id, values in grouped.items()
    }


def clustered_method_contrast(
    unit_rows,
    environment,
    proposed,
    comparator,
    metric,
    repetitions,
    seed,
):
    proposed_map = state_metric_map(
        unit_rows, environment, proposed, metric
    )
    comparator_map = state_metric_map(
        unit_rows, environment, comparator, metric
    )
    state_ids = sorted(set(proposed_map) & set(comparator_map))
    sign = -1.0 if metric in LOWER_IS_BETTER else 1.0
    differences = np.asarray(
        [
            sign * (
                proposed_map[state_id]
                - comparator_map[state_id]
            )
            for state_id in state_ids
        ],
        dtype=np.float64,
    )
    if not len(differences):
        return {
            "environment": environment,
            "proposed": proposed,
            "comparator": comparator,
            "metric": metric,
            "positive_means_proposed_better": True,
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": int(repetitions),
        }
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions, dtype=np.float64)
    for index in range(repetitions):
        sampled = rng.integers(0, len(differences), len(differences))
        draws[index] = float(np.mean(differences[sampled]))
    return {
        "environment": environment,
        "proposed": proposed,
        "comparator": comparator,
        "metric": metric,
        "positive_means_proposed_better": True,
        "estimate": float(np.mean(differences)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(differences)),
        "n_bootstrap": int(repetitions),
    }


def clustered_error_ratio(
    unit_rows,
    environment,
    proposed,
    comparator,
    repetitions,
    seed,
):
    proposed_map = state_metric_map(
        unit_rows,
        environment,
        proposed,
        "ordinary_latent_rmse",
    )
    comparator_map = state_metric_map(
        unit_rows,
        environment,
        comparator,
        "ordinary_latent_rmse",
    )
    state_ids = sorted(set(proposed_map) & set(comparator_map))
    proposed_value = np.asarray(
        [proposed_map[state_id] for state_id in state_ids]
    )
    comparator_value = np.asarray(
        [comparator_map[state_id] for state_id in state_ids]
    )
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(state_ids), len(state_ids))
        draws[index] = np.mean(proposed_value[sampled]) / max(
            np.mean(comparator_value[sampled]), 1e-12
        )
    return {
        "environment": environment,
        "proposed": proposed,
        "comparator": comparator,
        "metric": "ordinary_latent_rmse_ratio",
        "estimate": float(
            np.mean(proposed_value)
            / max(np.mean(comparator_value), 1e-12)
        ),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "margin": float(LATENT_ERROR_RATIO_MARGIN),
        "n_clusters": int(len(state_ids)),
        "n_bootstrap": int(repetitions),
    }


def analyze_recurrent_results(unit_rows):
    summary_rows = development_summary(unit_rows)
    contrast_rows = []
    ratio_rows = []
    proposed = "counterfactual_recurrent"
    comparators = [
        "base_world_model",
        "absolute_residual",
        "independent_delta_control",
    ]
    primary_metrics = [
        "normalized_regret",
        "weighted_pairwise_accuracy",
        "normalized_paired_latent_rmse",
    ]
    for environment in ENVIRONMENT:
        for comparator in comparators:
            for metric in primary_metrics:
                contrast_rows.append(
                    clustered_method_contrast(
                        unit_rows,
                        environment,
                        proposed,
                        comparator,
                        metric,
                        BOOTSTRAP_REPS,
                        stable_seed(
                            "contrast",
                            environment,
                            comparator,
                            metric,
                        ),
                    )
                )
        ratio_rows.append(
            clustered_error_ratio(
                unit_rows,
                environment,
                proposed,
                "base_world_model",
                BOOTSTRAP_REPS,
                stable_seed("ratio", environment),
            )
        )

    gates = {}
    for environment in ENVIRONMENT:
        base = {
            row["metric"]: row
            for row in contrast_rows
            if row["environment"] == environment
            and row["comparator"] == "base_world_model"
        }
        absolute = {
            row["metric"]: row
            for row in contrast_rows
            if row["environment"] == environment
            and row["comparator"] == "absolute_residual"
        }
        independent = {
            row["metric"]: row
            for row in contrast_rows
            if row["environment"] == environment
            and row["comparator"] == "independent_delta_control"
        }
        ratio = next(
            row for row in ratio_rows if row["environment"] == environment
        )
        base_planning = bool(
            base["normalized_regret"]["low"] > 0.0
            and base["weighted_pairwise_accuracy"]["low"] > 0.0
        )
        latent_noninferiority = bool(
            ratio["high"] <= LATENT_ERROR_RATIO_MARGIN
        )
        specificity = bool(
            (
                absolute["normalized_regret"]["low"] > 0.0
                or absolute["weighted_pairwise_accuracy"]["low"] > 0.0
            )
            and (
                independent["normalized_regret"]["low"] > 0.0
                or independent[
                    "weighted_pairwise_accuracy"
                ]["low"]
                > 0.0
            )
        )
        gates[environment] = {
            "base_planning_pass": base_planning,
            "ordinary_latent_noninferiority_pass": latent_noninferiority,
            "complete_base_gate_pass": bool(
                base_planning and latent_noninferiority
            ),
            "specificity_pass": specificity,
        }

    complete_count = sum(
        value["complete_base_gate_pass"] for value in gates.values()
    )
    if PIPELINE_FAILED:
        status = "INCONCLUSIVE"
    elif complete_count == len(ENVIRONMENT) and all(
        value["specificity_pass"] for value in gates.values()
    ):
        status = "RECURRENT_COUNTERFACTUAL_CANDIDATE_READY"
    elif complete_count == len(ENVIRONMENT):
        status = "RECURRENT_GAIN_NOT_SPECIFIC"
    elif complete_count == 1:
        status = "MIXED_RECURRENT_SIGNAL"
    else:
        status = "NO_RECURRENT_DEVELOPMENT_GAIN"

    decision = {
        "status": status,
        "evidence_status": EVIDENCE_STATUS,
        "proposed_method": proposed,
        "primary_baseline": "base_world_model",
        "same_architecture_baseline": "absolute_residual",
        "specificity_baseline": "independent_delta_control",
        "latent_error_ratio_margin": LATENT_ERROR_RATIO_MARGIN,
        "environment_gates": gates,
        "layerwise_audit_gate": json.loads(
            (OUT / "layerwise_audit_gate.json").read_text()
        ),
        "interpretation_boundary": (
            "Exploratory development on previously inspected Stage 5 tasks; "
            "a positive result nominates a frozen method for new tasks."
        ),
    }
    write_csv(OUT / "metrics_summary.csv", summary_rows)
    write_csv(OUT / "method_contrasts.csv", contrast_rows)
    write_csv(OUT / "latent_error_noninferiority.csv", ratio_rows)
    write_json(OUT / "stage7_development_decision.json", decision)
    return summary_rows, contrast_rows, ratio_rows, decision


if not PIPELINE_FAILED:
    try:
        UNIT_ROWS, ACTION_ROWS = evaluate_recurrent_methods()
        (
            METRICS_SUMMARY_ROWS,
            METHOD_CONTRAST_ROWS,
            LATENT_RATIO_ROWS,
            STAGE7_DECISION,
        ) = analyze_recurrent_results(UNIT_ROWS)
        (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")
    except Exception:
        record_failure("recurrent_evaluation_and_analysis")


In [ ]:
# Phase E — compact diagnostic plots.


def make_stage7_plots():
    if not (OUT / "metrics_summary.csv").exists():
        return
    import pandas as pd

    summary = pd.read_csv(OUT / "metrics_summary.csv")
    audit = pd.read_csv(OUT / "layerwise_audit.csv")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for axis, metric, title in [
        (axes[0], "normalized_regret", "Development normalized regret"),
        (
            axes[1],
            "weighted_pairwise_accuracy",
            "Development weighted pairwise accuracy",
        ),
    ]:
        pivot = summary.pivot(
            index="method", columns="environment", values=metric
        )
        pivot.plot(kind="bar", ax=axis)
        axis.set_title(title)
        axis.set_xlabel("")
        axis.grid(axis="y", alpha=0.25)
        axis.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "recurrent_planning_by_method.png", dpi=170)
    plt.close(fig)

    grouped = (
        audit.groupby(["environment", "layer_index", "layer"], as_index=False)[
            "development_effect_r2"
        ]
        .mean()
        .sort_values(["environment", "layer_index"])
    )
    fig, axis = plt.subplots(figsize=(9, 4.8))
    for environment in ENVIRONMENT:
        values = grouped[grouped["environment"] == environment]
        axis.plot(
            values["layer_index"],
            values["development_effect_r2"],
            marker="o",
            label=environment,
        )
    axis.axhline(0.0, color="black", linewidth=1, alpha=0.6)
    axis.set_xticks(range(1, len(AUDIT_LAYERS) + 1), AUDIT_LAYERS, rotation=25)
    axis.set_ylabel("Development physical-effect R²")
    axis.set_title("Layerwise counterfactual decodability")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "layerwise_effect_decodability.png", dpi=170)
    plt.close(fig)


if not PIPELINE_FAILED:
    try:
        make_stage7_plots()
    except Exception:
        record_failure("stage7_plots")


In [ ]:
# Phase F — package all non-cache outputs and download one result bundle.


def package_stage7_results():
    result_zip = Path("/content/stage7_result_bundle.zip")
    excluded_roots = {
        str(INTERMEDIATE.resolve()),
        str(CACHE_ROOT.resolve()),
    }
    files = []
    for path in OUT.rglob("*"):
        if not path.is_file():
            continue
        resolved = str(path.resolve())
        if any(
            resolved == root or resolved.startswith(root + os.sep)
            for root in excluded_roots
        ):
            continue
        files.append(path)
    manifest = [
        {
            "path": str(path.relative_to(OUT)),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        }
        for path in sorted(files)
    ]
    write_json(
        OUT / "result_zip_manifest.json",
        {
            "run_signature": RUN_SIGNATURE,
            "pipeline_failed": bool(PIPELINE_FAILED),
            "files": manifest,
        },
    )
    files = sorted(
        {
            *files,
            OUT / "result_zip_manifest.json",
            OUT / "FAILURE_TRACE.txt",
        }
    )
    with zipfile.ZipFile(
        result_zip, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in files:
            if path.exists():
                archive.write(path, path.relative_to(OUT))
    print(f"RESULT_ZIP: {result_zip}")
    print(
        "RUN_STATUS:",
        "FAILED" if PIPELINE_FAILED else "SUCCESS",
    )
    if DOWNLOAD_RESULTS:
        from google.colab import files as colab_files

        colab_files.download(str(result_zip))
    return result_zip


try:
    RESULT_ZIP = package_stage7_results()
except Exception:
    record_failure("stage7_packaging")
    raise
